# Trabajo final · Data Analyst · Sabor Express

**Delivery de comida** · Empresa ficticia. Los datos son sintéticos y **se generan por código en este notebook**: ningún resultado representa una situación real de ARCA.

## La pregunta oficial

> **¿La suscripción Express+ hace que los clientes gasten más?**

**Decisión que habilita:** Si conviene invertir más en la campaña de suscripción.

Antes de analizar, **verificá si los datos sostienen la premisa de la pregunta**. A veces no la sostienen, y decirlo es un hallazgo.

Las instrucciones completas (qué se entrega, rúbrica y defensa) están en el PDF **«Trabajo final · Data Analyst · Instrucciones»**. Este notebook es tu lugar de trabajo: vas del **SQL al análisis** sin salir de acá.

| Sección | Módulo del programa |
| --- | --- |
| 2 | SQL y PL/SQL |
| 3 | Estadística |
| 4 | Python: limpieza |
| 5 | Proceso de análisis |
| 6 | Superset |
| 7 | Analítica de negocio |
| 8 | Bitácora y entrega |

## Paso 0 · Instalación

Corré esta celda primero. Si falta alguna librería, la instala en el mismo Python que ejecuta este notebook.

In [ ]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib"}
print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")

## 1 · La base de Sabor Express

La primera celda **genera la base** con código comprimido a propósito: no hace falta leerla, y leerla te quita la mitad del trabajo. La segunda la carga en SQLite y te deja tres herramientas: `sql(...)` para consultar, `esquema()` para ver las tablas y `anotar(...)` para dejar registrado cada problema de calidad que encuentres.

In [ ]:
import zlib, base64
_BLOB = (
    "eNp1V9tu3DYQffdXEAICSI2W0d5c167ykiAoUMA16rwUwkLgStw1E4kiSMmJHfjfe4bci7y2jfWaEsnhmZlzZmjVms72TA+t"
    "eWDCMW1SZoSuMcTH1Ge13LDSDZXqYpsyJ62SKbsXVgndS4fFyeUZw49jeZjlVWce4oQL1z8YGXfrb7LqkyvWYoHlFra7Nm6k"
    "jl2SsD+ZCbuLdoX54mA4vk+ZTdims+yeKe0XrK6Ylf1g8XQWcLXiYXCldEYQPr/lcr+muOdN90PaOAFePhizG7L3LGJRSl8Y"
    "3q8KyxVO3Err4ixli2QVjNdiXjqx7mwpfxornYudlHU+z+Y7jy0Aa7PziGOLGJq+tHrrF8LjayxYZFl2xR47LShARfRJ6t52"
    "dPw14i5pcDtY+vOpc72gwQ2iuMGviOCwqHoft6bb6s62oiGMfLlM2TVOMMjMOsCoGmVinp3DJz5bst9YTFs/kAHeSqHjhE0Y"
    "P0cEeDZLaXTFwuZDUq59Psik9w8WJaUCS0zNP4tefLGilfGvSNXlbjK6ZMUm+vuXusyW9VPk86UoXzC5lbC4gkPkPRZaXt11"
    "qpKxjwYcAHnygs/JoRlBml7Q19J/rQA02sjqTpSi6Wl7DBBfVSsRpdbE0SybLSbZFJ8ogc+Y7Luyx3wtsT5+ntT5uQ9Yygat"
    "+jz6HCUJd73d0Po4evff5F07eVdHyVPyzPNih8ANrrLKgGM6WoVw3BLVXVy8RLWcZPMRqq8HSLV4cDlQxXXyxvFMbaAj2ThJ"
    "GdVC+3giVDWF9BEJRm5A8me+TZcZ+ZasAnYwtX81YzQhBisOWfv3ltI2fyVtU1idTX3u7LAGXUfJK6Ib9fgoHDH1L9GuB7sd"
    "pAvPt4O7U57CwlrVNJ7Ot6IZarFuPNdvQHIAiVZ0Qpa8TQ6afQJFN6oJyoEWFP5m3knC+31Ms6BIP7Xj6zEp+2zyF8ksvq9C"
    "zIc1hiHw152WR2M4B2ZGkZmNjqIfpdXpeTsSjKmJVMh/Nhsn+7jtdH/nciwm/XY+WcCsUHq73p/OoEZv9yPNXLFGUOnkFyRq"
    "qJmQQt1Tns0JuzfhkU9hUGsvaNMp5zodY2vyDC45VB7d0frEHe8Se5+z6RW766yANWLsITu7KIAgs0VC+gVPkWvxEBdQMER2"
    "DrFhcv+Z+5fQ9R8p+x1vEO8PbHmeAGqtjtbHfJ5d0OzGz6lXNYSdKdAN1uWEMWWt0gMSnL80dp5Bay89bA1FvelEH1MF7wZd"
    "x+Mae8GpFC2WyS7Si9NIp2wyTfb05ALNBRbiTXRDkjqHpNJDEeHHaonUpUemkyD5c2HC7iYleCkLnh2MkCyIrsEZc6pwjwPL"
    "u2ZotcsLEryRtap9qxnV6/A0rgWHOksn+nl/JfAT+1d0ellFK2IYjqbubXZIigi5UV1pxLbzxXFUKXphv8nQ1CTO6NW9h7NG"
    "aZC9hGmEQ4cusKCyP5/7VkDHwG6FeHel1PcqGEamftxJK2PDAx6W5+NuiaJxkVFBnGYZ6e6kBdCUhuXTfjrfne2nke/RMV4B"
    "UOGMCiKfY9n0ZMVbQPiMgAQ/eiUR0ZIav9yKEmTduxOI1x8iSVmpT5w9dGdN3RlURAAroSvZCJ/cNxaOML48n31kdIMIXTcj"
    "NkdWVigWwWS0W4oH4nhvfMTQOlE9Wn//oCatg3fGdkahhJ9k6EgKH5xR+kNnS4/+G75jHELbm72yiACiURtViUPffSsoy/Rw"
    "AxrJeZffBZ/TzSebXpCYXw3HBOxIwn7qSRwVCNlD6Ur2cEey4zU8jo9Ke0UjBvSqioMQNDVpCF+aRlQy/yJQQ5KR0J6rBvGd"
    "zJZZRi1yScaC1qtOVyhXBd3PuUOfaagbYXGIRAnq9DL3N0/sVMRuWSpdy5/5VzvIZL9nY0WVT1/bxlEVZB/2xORk2Hg1ug75"
    "Zk1wj/8RPKtQ6cmVnOi12N9L/K38V7TfgKa/H3r+HSoSzdAj3oYSRi/M09nZVmoUDbp2v7ibn/0PoF7kpg=="
)
exec(zlib.decompress(base64.b64decode(_BLOB)).decode("utf-8"))   # define generar(): código comprimido a propósito

In [ ]:
import sqlite3, matplotlib.pyplot as plt
pd.options.display.width = 200; pd.options.display.max_columns = 40; pd.options.display.float_format = "{:,.3f}".format
t = generar(); tablas = {k: v for k, v in t.items() if not k.startswith("_")}
con = sqlite3.connect(":memory:")
for nombre, df in tablas.items(): df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, con, index=False)

def sql(consulta, parametros=None):
    """Corre una consulta SQL sobre la base y devuelve un DataFrame."""
    return pd.read_sql_query(consulta, con, params=parametros)

def esquema():
    """Lista las tablas, sus columnas y sus filas."""
    for nombre, df in tablas.items(): print(f"{nombre}  ({len(df):,} filas)".replace(",", "."), "\n   ", ", ".join(df.columns))

bitacora_calidad = pd.DataFrame(columns=["problema", "dónde", "filas afectadas", "qué decidimos y por qué"])
def anotar(problema, donde, filas, decision):
    """Deja anotado un problema de calidad y la decisión tomada. Es parte de la trazabilidad."""
    bitacora_calidad.loc[len(bitacora_calidad)] = [problema, donde, filas, decision]; return bitacora_calidad

def exportar_superset(carpeta="superset"):
    """Guarda la base como archivo SQLite y un CSV por tabla, para conectar Apache Superset."""
    os.makedirs(carpeta, exist_ok=True); salida = sqlite3.connect(f"{carpeta}/base.db")
    for nombre, df in tablas.items(): df.to_csv(f"{carpeta}/{nombre}.csv", index=False); df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, salida, index=False, if_exists="replace")
    salida.close(); print(f"Listo: {carpeta}/base.db y un CSV por tabla. Conectá Superset a ese archivo."); return carpeta
import os
esquema()

Los datos tienen los problemas de un dato real. Encontrarlos, decidir qué hacer con cada uno y dejarlo escrito es parte de la evidencia.

## 2 · SQL y PL/SQL

Tres consultas. Guardalas en esta sección: son la evidencia del módulo.

1. El gasto anual por cliente según si está suscripto (JOIN de `pedidos` con `clientes`).
2. Los pedidos de cada suscriptor antes y después de su `fecha_suscripcion`.
3. Los reclamos según el tiempo de entrega.

SQLite no tiene PL/SQL. Si querés encapsular una lógica, escribila como una función de Python que ejecuta SQL.

In [ ]:
sql("""
-- Consulta 1
SELECT 1
""")

In [ ]:
sql("""
-- Consulta 2
SELECT 1
""")

In [ ]:
sql("""
-- Consulta 3 (control de integridad)
SELECT 1
""")

## 3 · Estadística

Para la variable principal (`tiempo_entrega_min`): **media, mediana, dispersión y distribución**. Decidí cuál resume mejor y escribí por qué.

In [ ]:
# Tu análisis estadístico

## 4 · Python: limpieza

Traé las tablas a pandas (`df = sql("SELECT * FROM tabla")`) y documentá la calidad: **faltantes, duplicados, valores imposibles y categorías inconsistentes**. Limpiá **sin borrar en silencio**: cada decisión se anota con `anotar(...)`.

In [ ]:
# Tu limpieza
# anotar("problema", "tabla.columna", 0, "qué decidimos y por qué")

In [ ]:
bitacora_calidad

## 5 · Proceso de análisis

**Verificá la premisa** de la pregunta oficial antes de seguir. Después armá al menos dos variables nuevas (fecha del pedido, hora del día, antes o después de suscribirse, gasto previo del cliente).

In [ ]:
# Tu verificación de la premisa y tus variables nuevas

## 6 · Dashboard en Superset

Exportá la base y conectá Apache Superset al archivo. Armá **4 a 6 gráficos, uno por pregunta**, con un orden de lectura y las definiciones a la vista. Si no podés usar Superset, trabajá con gráficos de este notebook.

In [ ]:
exportar_superset()

In [ ]:
# Si no usás Superset, tus gráficos con matplotlib

## 7 · Analítica de negocio

**KPIs** con su denominador: gasto por cliente, ticket medio, pedidos por mes, tiempo de entrega, tasa de reclamos y propina.

**Segmentación:** suscriptores contra no suscriptores, zona, hora y medio de pago.

Un **insight** relevante, una **limitación** y una **recomendación** proporcional a la evidencia.

In [ ]:
# Tus KPIs y tu segmentación

## 8 · Bitácora y entrega

Completá esta bitácora editando la celda. Con ella, otra persona tiene que poder rehacer tus números.

**Equipo:** ____  **Integrantes y roles:** ____

**Pregunta y premisa:** ¿los datos sostienen la premisa? ____

| KPI | Fórmula | Universo | Período | Estados o faltantes |
| --- | --- | --- | --- | --- |
|  |  |  |  |  |

**Hechos** (lo que muestran los datos): ____

**Inferencias** (lo que creemos que significa): ____

**Lo que contradice nuestra hipótesis:** ____

**Recomendación** (proporcional a la evidencia): ____

**Límites y siguiente análisis:** ____

### Antes de entregar

- [ ] Reinicié el kernel y corrí todo el notebook de punta a punta.
- [ ] Mi pregunta termina en una decisión y nombra quién la toma.
- [ ] Verifiqué la premisa de la pregunta.
- [ ] Cada KPI declara fórmula, universo, período y tratamiento de estados.
- [ ] Ningún porcentaje se muestra sin decir sobre cuántos casos.
- [ ] La limpieza está anotada: qué encontré, qué hice y cuántas filas afectó.
- [ ] Los hechos están separados de las inferencias y no prometo causalidad.
- [ ] Mostré al menos un resultado que no me conviene.
- [ ] Hay una limitación y un siguiente análisis.